In [1]:
import numpy as np
from jax import numpy as jnp
import jax

from blueprint.qubits import TunableTransmon, AnharmonicOscillator
from blueprint.devices import Device

import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = '0.30' # for 50% of total GPU memory

jax.config.update('jax_default_device', jax.devices('cpu')[0])
# jax.config.update('jax_default_device', jax.devices('cuda')[1])
jax.config.update('jax_enable_x64', True) # Double
dtype = jnp.complex128
# jax.config.update('jax_enable_x64', False) # Simple
# dtype = jnp.complex64

CUDA backend failed to initialize: Found CUDA version 12010, but JAX was built against version 12030, which is newer. The copy of CUDA that is installed must be at least as new as the version against which JAX was built. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


In [2]:
jax.devices

<function jax._src.xla_bridge.devices(backend: 'str | xla_client.Client | None' = None) -> 'list[xla_client.Device]'>

In [3]:
label = "Q1"
charging_energy = 0.24193246329355922
josephson_energy = 16.861934403863096
charge_cutoff = 100
offset_charge = 0.0
dim = 3

transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff
)
transmon.diagonalize(dim)

label = "Q2"
charging_energy = 0.23958546598194805
josephson_energy = 18.062900398155875
charge_cutoff = 100
offset_charge = 0.0
dim = 3

other_transmon = TunableTransmon(
    label=label,
    charging_energy=charging_energy,
    josephson_energy=josephson_energy,
    offset_charge=offset_charge,
    charge_cutoff=charge_cutoff
)
other_transmon.diagonalize(dim)

# Comparing with the old implementation

In [4]:

EJ_GHz = transmon.max_josephson_energy
EC_GHz = transmon.charging_energy
ng = transmon.offset_charge
n_charge_states = transmon.charge_cutoff
dim = transmon.dim

dtype = jnp.complex64

charge_op = jnp.diag(jnp.arange(-1 * n_charge_states, n_charge_states + 1))
dim_charge = 2*n_charge_states + 1
ones_offdiag = jnp.ones((dim_charge - 1,), dtype=dtype)
cosphi = jnp.diag(0.5 * ones_offdiag, k=1) + jnp.diag(0.5 * ones_offdiag, k=-1)
I = jnp.identity(dim_charge)
# H = 4 Ec (n - ng)^2 - Ej cos(phi)
H = 4 * EC_GHz * (charge_op - ng * I) @ (charge_op - ng * I) - EJ_GHz * cosphi
[D, V] = jnp.linalg.eigh(H)

eigvals = jnp.array(D[:dim] - D[0], dtype=dtype)

eigkets = V[:, :dim].T # Shape (dim, dim_charge)
for iket, eket in enumerate(eigkets):
    index = jnp.argmax(abs(eket))
    eigkets.at[iket].set(eigkets[iket] * jnp.exp(-1j * jnp.angle(eket[index])))
eigkets = eigkets.T

H_diag = jnp.diag(eigvals)
# Truncate to `dim` levels only using this truncated eigenvectors matrix.
# trunc_V = V[:, :dim]
trunc_V = eigkets
n_diag = jnp.conj(trunc_V).T @ charge_op @ trunc_V
cosphi_diag = jnp.conj(trunc_V).T @ cosphi @ trunc_V

In [5]:
comp_charge_op = transmon._get_charge_op()
assert jnp.allclose(charge_op, comp_charge_op)

comp_cosphi_op = transmon._get_cosphi_op()
assert jnp.allclose(cosphi, comp_cosphi_op)

comp_hamil = transmon._get_hamiltonian()
assert jnp.allclose(H, comp_hamil)

comp_eig_vals, comp_eig_vecs = transmon.eigenstates()
assert jnp.allclose(D, comp_eig_vals)
assert jnp.allclose(V, comp_eig_vecs)

In [6]:
comp_transform = transmon._transform
jnp.allclose(trunc_V, comp_transform)

comp_diag_charge_op = transmon.get_charge_op()
assert jnp.allclose(n_diag, comp_diag_charge_op)

comp_diag_cosphi_op = transmon.get_cosphi_op()
assert jnp.allclose(cosphi_diag, comp_diag_cosphi_op)

comp_diag_hamil = transmon.get_hamiltonian()
assert jnp.allclose(H_diag, comp_diag_hamil)

In [7]:
device = Device((transmon, other_transmon))

In [8]:
device.add_capacative_coupling(qubit_labels=("Q1", "Q2"), coupler_label="G1", prefactor=0.0025)

In [9]:
hamiltonian = device.get_hamiltonian()

# Example using the approximate Transmon implementation 

In [10]:
label = "transmon"
frequency: float = 5.0
anharmonicity: float = -0.3
ext_flux: float = 0.0
dim: int = 5

transmon = AnharmonicOscillator(
    label=label,
    frequency=frequency,
    anharmonicity=anharmonicity,
    ext_flux=ext_flux,
    dim=dim
)

In [16]:
assert np.allclose(transmon.fundamental_frequency, transmon.frequency)

In [15]:
assert np.allclose(transmon.abs_anharmonicity, transmon.anharmonicity)

In [17]:
print(transmon.get_hamiltonian())

[[ 0.   0.   0.   0.   0. ]
 [ 0.   5.   0.   0.   0. ]
 [ 0.   0.   9.7  0.   0. ]
 [ 0.   0.   0.  14.1  0. ]
 [ 0.   0.   0.   0.  18.2]]
